# Plotly Study Guide

Interactive charts for the web. Run in Jupyter — charts render inline with full interactivity.

## 1. 1. Plotly Express Basics

plotly.express (px) creates interactive charts in one line. Charts are HTML/JavaScript — hover, zoom, and pan by default.

**First chart: px.scatter**

In [ ]:
import plotly.express as px

# Built-in dataset
df = px.data.gapminder().query("year == 2007")

fig = px.scatter(
    df, x='gdpPercap', y='lifeExp',
    size='pop', color='continent',
    hover_name='country',
    log_x=True,
    title='GDP per Capita vs Life Expectancy (2007)',
    labels={'gdpPercap': 'GDP per Capita (log)', 'lifeExp': 'Life Expectancy'}
)
fig.show()

**px.line — time series**

In [ ]:
import plotly.express as px

df = px.data.gapminder().query("continent == 'Europe'")

fig = px.line(
    df, x='year', y='lifeExp',
    color='country',
    hover_name='country',
    title='Life Expectancy in Europe Over Time',
    labels={'lifeExp': 'Life Expectancy', 'year': 'Year'}
)
fig.update_layout(showlegend=False)   # too many countries for legend
fig.show()

### Real-World: Sales KPI Interactive Dashboard

> A product manager builds a quick interactive scatter to explore the relationship between marketing spend and revenue by market.

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

np.random.seed(42)
markets = ['US','UK','DE','FR','JP','BR','IN','AU','CA','MX']
df = pd.DataFrame({
    'market':       markets,
    'ad_spend':     np.random.uniform(50, 500, 10).round(1),
    'revenue':      np.random.uniform(200, 2000, 10).round(1),
    'customers':    np.random.randint(500, 10000, 10),
    'region':       ['Americas','Europe','Europe','Europe',
                     'Asia','Americas','Asia','Pacific','Americas','Americas'],
})
df['roi'] = (df['revenue'] / df['ad_spend']).round(2)

fig = px.scatter(
    df, x='ad_spend', y='revenue',
    size='customers', color='region',
    text='market',
    hover_data=['roi', 'customers'],
    title='Marketing Spend vs Revenue by Market',
    labels={'ad_spend': 'Ad Spend ($K)', 'revenue': 'Revenue ($K)'},
    size_max=50
)
fig.update_traces(textposition='top center', textfont_size=10)
fig.update_layout(height=450)
fig.show()

## 2. 2. Bar & Pie Charts

px.bar for categorical comparisons; px.pie and px.sunburst for part-to-whole. All support hover, faceting, and animation.

**px.bar with facets**

In [ ]:
import plotly.express as px

df = px.data.tips()

fig = px.bar(
    df, x='day', y='total_bill',
    color='sex', barmode='group',
    facet_col='time',
    title='Total Bill by Day, Gender, and Meal Time',
    labels={'total_bill': 'Total Bill ($)', 'day': 'Day'},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_layout(height=400)
fig.show()

**px.pie and px.sunburst**

In [ ]:
import plotly.express as px
import pandas as pd

# Pie chart
df_pie = pd.DataFrame({
    'channel':  ['Organic', 'Paid Search', 'Social', 'Email', 'Direct'],
    'sessions': [35, 25, 20, 12, 8],
})
fig1 = px.pie(df_pie, names='channel', values='sessions',
              title='Traffic by Channel', hole=0.4)
fig1.show()

# Sunburst — hierarchical
df_sun = px.data.gapminder().query("year==2007 and continent in ['Europe','Americas']")
fig2 = px.sunburst(df_sun, path=['continent','country'],
                   values='pop', color='lifeExp',
                   color_continuous_scale='RdYlGn',
                   title='Population & Life Expectancy')
fig2.show()

### Real-World: Revenue Breakdown by Product & Region

> A CFO uses an interactive sunburst chart to drill down from region → product category → SKU in the quarterly review.

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

np.random.seed(5)
regions    = ['North America', 'Europe', 'Asia Pacific']
categories = ['Electronics', 'Clothing', 'Home', 'Food']
skus_per   = 3

rows = []
for region in regions:
    for cat in categories:
        for sku_i in range(1, skus_per + 1):
            rows.append({
                'region':   region,
                'category': cat,
                'sku':      f'{cat[:3]}-{sku_i:03d}',
                'revenue':  round(np.random.uniform(50, 500), 1),
            })
df = pd.DataFrame(rows)

fig = px.sunburst(
    df, path=['region', 'category', 'sku'],
    values='revenue',
    color='revenue',
    color_continuous_scale='Blues',
    title='Q2 2024 Revenue Drill-Down (Region → Category → SKU)',
)
fig.update_layout(height=550, coloraxis_showscale=False)
fig.show()

## 3. 3. Histogram & Box Plot

px.histogram and px.box create interactive distribution charts. Hover shows exact statistics; click legend to toggle groups.

**px.histogram with marginal**

In [ ]:
import plotly.express as px

df = px.data.tips()

fig = px.histogram(
    df, x='total_bill',
    color='time', barmode='overlay',
    marginal='box',       # adds mini box plot on top
    opacity=0.7,
    nbins=25,
    title='Total Bill Distribution by Meal Time',
    labels={'total_bill': 'Total Bill ($)'},
    color_discrete_sequence=['#636EFA','#EF553B']
)
fig.show()

**px.box and px.violin**

In [ ]:
import plotly.express as px
import pandas as pd

df = px.data.tips()

fig1 = px.box(df, x='day', y='tip', color='smoker',
              notched=True,
              title='Tip Distribution — Box Plot (notched)',
              labels={'tip': 'Tip ($)'})
fig1.show()

fig2 = px.violin(df, x='day', y='total_bill', color='sex',
                 box=True, points='outliers',
                 title='Total Bill — Violin + Box',
                 labels={'total_bill': 'Total Bill ($)'})
fig2.show()

### Real-World: A/B Test Distribution Explorer

> A data scientist uses interactive histograms to explore the full distribution of test results across experiment variants.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from scipy import stats

np.random.seed(42)
control = np.random.normal(45.0, 12, 500)
variant = np.random.normal(49.5, 11, 500)

df = pd.DataFrame({
    'revenue':  np.concatenate([control, variant]),
    'group':    ['Control']*500 + ['Variant B']*500,
})

t, p = stats.ttest_ind(control, variant)

fig = px.histogram(
    df, x='revenue', color='group',
    barmode='overlay', opacity=0.65, nbins=35,
    marginal='violin',
    color_discrete_map={'Control':'#636EFA','Variant B':'#EF553B'},
    title=f'A/B Test Revenue Distribution  |  p-value={p:.4f} {"✓ Significant" if p<0.05 else "✗ Not significant"}',
    labels={'revenue': 'Revenue ($)'},
)
fig.add_vline(x=control.mean(), line_dash='dash', line_color='#636EFA',
              annotation_text=f'Control μ={control.mean():.1f}')
fig.add_vline(x=variant.mean(), line_dash='dash', line_color='#EF553B',
              annotation_text=f'Variant μ={variant.mean():.1f}')
fig.update_layout(height=450)
fig.show()

## 4. 4. Heatmap & Scatter Matrix

px.imshow renders 2D matrices as interactive heatmaps. px.scatter_matrix creates an interactive pairplot grid.

**px.imshow — correlation heatmap**

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

np.random.seed(42)
features = ['Revenue', 'Cost', 'Margin', 'Customers', 'NPS']
n = 200
data = np.random.randn(n, 5)
# Add correlations
data[:, 2] = 0.8*data[:,0] - 0.6*data[:,1] + np.random.randn(n)*0.3
data[:, 4] = 0.5*data[:,2] + np.random.randn(n)*0.7

df   = pd.DataFrame(data, columns=features)
corr = df.corr().round(2)

fig = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Feature Correlation Matrix'
)
fig.update_layout(height=450)
fig.show()

**px.scatter_matrix**

In [ ]:
import plotly.express as px

df = px.data.iris()

fig = px.scatter_matrix(
    df,
    dimensions=['sepal_length','sepal_width','petal_length','petal_width'],
    color='species',
    symbol='species',
    title='Iris Dataset — Interactive Scatter Matrix',
    labels={col: col.replace('_',' ') for col in df.columns}
)
fig.update_traces(diagonal_visible=False, showupperhalf=False)
fig.update_layout(height=600)
fig.show()

### Real-World: Operations Metrics Heatmap

> An operations manager visualizes a week × hour activity heatmap to identify peak load periods for staffing.

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

np.random.seed(7)
hours = list(range(24))
days  = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

# Simulate ticket volumes: high weekday business hours
data = np.random.poisson(3, (7, 24)).astype(float)
data[0:5, 9:18] += np.random.poisson(12, (5, 9))   # weekday 9-18
data[0:5, 0:6]  *= 0.3                              # overnight low
data[5:7, :]    *= 0.5                              # weekend lower

df = pd.DataFrame(data.round(0).astype(int),
                  index=days, columns=[f'{h:02d}:00' for h in hours])

fig = px.imshow(
    df,
    color_continuous_scale='YlOrRd',
    aspect='auto',
    title='Support Ticket Volume — Week × Hour Heatmap',
    labels=dict(x='Hour', y='Day', color='Tickets')
)
fig.update_xaxes(tickangle=45, tickmode='array',
                 tickvals=list(range(0,24,2)),
                 ticktext=[f'{h:02d}:00' for h in range(0,24,2)])
fig.update_layout(height=380)
fig.show()

## 5. 5. 3D Charts

Plotly renders true 3D scatter and surface plots in the browser. Drag to rotate, scroll to zoom.

**3D scatter plot**

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

np.random.seed(42)
df = pd.DataFrame({
    'x':       np.random.randn(300),
    'y':       np.random.randn(300),
    'z':       np.random.randn(300),
    'cluster': np.random.choice(['A','B','C'], 300),
    'size':    np.random.uniform(5, 20, 300),
})
# Separate clusters
df.loc[df.cluster=='A','x'] += 2
df.loc[df.cluster=='B','y'] += 2
df.loc[df.cluster=='C','z'] += 2

fig = px.scatter_3d(df, x='x', y='y', z='z',
                    color='cluster', size='size',
                    opacity=0.7,
                    title='3D Cluster Visualization')
fig.update_layout(height=500)
fig.show()

**3D surface plot**

In [ ]:
import plotly.graph_objects as go
import numpy as np

x = np.linspace(-3, 3, 60)
y = np.linspace(-3, 3, 60)
X, Y = np.meshgrid(x, y)
Z = np.sin(np.sqrt(X**2 + Y**2)) * np.exp(-0.1*(X**2+Y**2))

fig = go.Figure(data=[
    go.Surface(z=Z, x=X, y=Y,
               colorscale='Viridis',
               contours=dict(z=dict(show=True, usecolormap=True,
                                    highlightcolor='white', project_z=True)))
])
fig.update_layout(
    title='3D Surface: Damped Sinc Function',
    scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
    height=500
)
fig.show()

### Real-World: 3D Risk Surface for Options Pricing

> A quant visualizes how an option's price changes with underlying price and time-to-expiry using a 3D surface.

In [ ]:
import plotly.graph_objects as go
import numpy as np
from scipy.stats import norm

def black_scholes_call(S, K, T, r, sigma):
    # Avoid division by zero
    T = np.where(T < 1e-6, 1e-6, T)
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

K     = 100      # strike price
r     = 0.05     # risk-free rate
sigma = 0.20     # volatility

S_vals = np.linspace(70, 130, 50)   # underlying price
T_vals = np.linspace(0.02, 1.0, 50) # time to expiry (years)
S_grid, T_grid = np.meshgrid(S_vals, T_vals)
C_grid = black_scholes_call(S_grid, K, T_grid, r, sigma)

fig = go.Figure(data=[go.Surface(
    x=S_grid, y=T_grid, z=C_grid,
    colorscale='Plasma',
    colorbar=dict(title='Call Price ($)')
)])
fig.update_layout(
    title=f'Black-Scholes Call Option Price  (K={K}, σ={sigma})',
    scene=dict(
        xaxis_title='Underlying Price (S)',
        yaxis_title='Time to Expiry (T)',
        zaxis_title='Call Price ($)',
    ),
    height=520
)
fig.show()

## 6. 6. Subplots with make_subplots

make_subplots creates multi-panel figures. Mix chart types, share axes, and control spacing — all in one interactive figure.

**2×2 subplot grid**

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

np.random.seed(5)
x = np.linspace(0, 10, 100)

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=['Line','Bar','Scatter','Histogram'])

fig.add_trace(go.Scatter(x=x, y=np.sin(x), name='sin'), row=1, col=1)
fig.add_trace(go.Bar(x=['A','B','C','D'], y=[4,7,3,8], name='bar'), row=1, col=2)
fig.add_trace(go.Scatter(x=np.random.randn(100), y=np.random.randn(100),
                         mode='markers', name='scatter',
                         marker=dict(opacity=0.5)), row=2, col=1)
fig.add_trace(go.Histogram(x=np.random.randn(500), name='hist'), row=2, col=2)

fig.update_layout(title='Multi-Type Dashboard', height=500, showlegend=False)
fig.show()

**Shared x-axis — price + volume**

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd

np.random.seed(9)
dates  = pd.date_range('2024-01-01', periods=60, freq='B')
price  = 100 + np.cumsum(np.random.randn(60) * 1.5)
volume = np.random.randint(1000, 8000, 60)
colors = ['green' if price[i] >= price[i-1] else 'red' for i in range(len(price))]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    row_heights=[0.7, 0.3],
                    vertical_spacing=0.03)

fig.add_trace(go.Scatter(x=dates, y=price, name='Price',
                         line=dict(color='royalblue', width=2)), row=1, col=1)
fig.add_trace(go.Bar(x=dates, y=volume, name='Volume',
                     marker_color=colors, opacity=0.7), row=2, col=1)

fig.update_layout(title='Stock Price & Volume', height=500, showlegend=False)
fig.update_xaxes(rangeslider_visible=False)
fig.show()

### Real-World: Marketing Analytics Dashboard

> A growth team builds a 4-panel Plotly dashboard showing funnel, revenue trend, channel mix, and conversion by cohort.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd

np.random.seed(3)
months  = pd.date_range('2024-01', periods=12, freq='MS')
revenue = 100 + np.cumsum(np.random.randn(12)*8) + np.arange(12)*5
sessions= np.random.randint(8000, 20000, 12)
cvr     = revenue / sessions * 100

channels = ['Organic','Paid','Social','Email','Direct']
ch_vals  = [35, 25, 18, 12, 10]

funnel_stages  = ['Visit','Sign Up','Trial','Paid']
funnel_vals    = [10000, 3200, 1100, 420]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Revenue Trend ($K)', 'Traffic by Channel',
                    'Conversion Funnel', 'CVR vs Sessions'],
    specs=[[{},{}],[{},{}]]
)

# Revenue
fig.add_trace(go.Scatter(x=months, y=revenue.round(1), fill='tozeroy',
                         line=dict(color='#636EFA'), name='Revenue'), row=1, col=1)

# Channel pie — use domain trace
fig.add_trace(go.Pie(labels=channels, values=ch_vals, hole=0.35,
                     showlegend=False, textinfo='label+percent'), row=1, col=2)

# Funnel
fig.add_trace(go.Funnel(y=funnel_stages, x=funnel_vals,
                        marker_color=['#636EFA','#EF553B','#00CC96','#AB63FA'],
                        textinfo='value+percent initial'), row=2, col=1)

# CVR vs Sessions scatter
fig.add_trace(go.Scatter(x=sessions, y=cvr.round(3), mode='markers+text',
                         text=[m.strftime('%b') for m in months],
                         textposition='top center',
                         marker=dict(size=10, color='#FFA15A')), row=2, col=2)

fig.update_layout(title='Growth Dashboard — 2024', height=650)
fig.show()

## 7. 7. Customizing Layout & Traces

update_layout() controls the figure-level design. update_traces() modifies all traces at once. Both accept selector filtering.

**update_layout — theme and fonts**

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

df = px.data.gapminder().query("year == 2007 and continent == 'Europe'")

fig = px.scatter(df, x='gdpPercap', y='lifeExp',
                 size='pop', color='country',
                 hover_name='country', log_x=True)

fig.update_layout(
    title=dict(text='Europe 2007: GDP vs Life Expectancy',
               font=dict(size=16, color='white'), x=0.5),
    plot_bgcolor='#1a1a2e',
    paper_bgcolor='#16213e',
    font=dict(color='#e0e0e0'),
    xaxis=dict(gridcolor='#333', title='GDP per Capita (log)'),
    yaxis=dict(gridcolor='#333', title='Life Expectancy'),
    showlegend=False,
    height=450,
)
fig.show()

**update_traces and add_shape**

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

np.random.seed(1)
x = np.arange(1, 13)
y = 80 + np.cumsum(np.random.randn(12) * 5)

fig = px.line(x=x, y=y, markers=True,
              title='Monthly Revenue with Target Zone')

# Customise the line trace
fig.update_traces(
    line=dict(color='#00CC96', width=3),
    marker=dict(size=9, color='white',
                line=dict(color='#00CC96', width=2))
)

# Add target band (rectangle shape)
fig.add_hrect(y0=90, y1=110, fillcolor='rgba(100,200,100,0.1)',
              line_width=0, annotation_text='Target range')
fig.add_hline(y=100, line_dash='dot', line_color='gray',
              annotation_text='Target')

fig.update_layout(xaxis_title='Month', yaxis_title='Revenue ($K)')
fig.show()

### Real-World: Branded Executive KPI Chart

> A BI developer creates a dark-themed, branded interactive chart for the C-suite weekly review email.

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

np.random.seed(42)
months  = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
revenue = np.array([4.2,3.8,5.1,5.8,6.2,7.0,6.5,7.8,8.1,7.5,9.2,10.4])
target  = np.full(12, 6.0)
prev_yr = revenue * np.random.uniform(0.75, 0.95, 12)

fig = go.Figure()

# Previous year (subtle background)
fig.add_trace(go.Bar(x=months, y=prev_yr, name='2023',
                     marker_color='rgba(100,100,180,0.3)', showlegend=True))

# Current year bars
fig.add_trace(go.Bar(x=months, y=revenue, name='2024',
                     marker_color=['#00CC96' if r>=t else '#EF553B'
                                   for r,t in zip(revenue, target)]))

# Target line
fig.add_trace(go.Scatter(x=months, y=target, name='Target',
                         line=dict(color='white', dash='dash', width=2),
                         mode='lines'))

# YoY growth annotations
for i, (m, r, p) in enumerate(zip(months, revenue, prev_yr)):
    yoy = (r-p)/p*100
    fig.add_annotation(x=m, y=r+0.15, text=f'+{yoy:.0f}%',
                       font=dict(size=8, color='#aaa'), showarrow=False)

fig.update_layout(
    title=dict(text='2024 Monthly Revenue ($M) vs Target & Prior Year',
               font=dict(size=15, color='white'), x=0.5),
    barmode='overlay',
    plot_bgcolor='#111827', paper_bgcolor='#0f172a',
    font=dict(color='#cbd5e1'),
    xaxis=dict(gridcolor='#1e293b'),
    yaxis=dict(gridcolor='#1e293b', title='Revenue ($M)'),
    legend=dict(orientation='h', y=1.08),
    height=460,
)
fig.show()

## 8. 8. Animated Charts

animation_frame adds a play button to animate over a variable (e.g. year, month). Great for showing trends over time.

**Animated bubble chart**

In [ ]:
import plotly.express as px

# Classic animated gapminder chart
df = px.data.gapminder()

fig = px.scatter(
    df, x='gdpPercap', y='lifeExp',
    animation_frame='year',
    animation_group='country',
    size='pop', color='continent',
    hover_name='country',
    log_x=True, size_max=55,
    range_x=[100, 100000], range_y=[25, 90],
    title='World Development 1952–2007',
    labels={'gdpPercap': 'GDP per Capita', 'lifeExp': 'Life Expectancy'},
)
fig.update_layout(height=520)
fig.show()

**Animated bar chart race**

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

np.random.seed(42)
products = ['Widget', 'Gadget', 'Doohickey', 'Gizmo', 'Thingamajig']
quarters = ['Q1','Q2','Q3','Q4']

rows = []
sales = {p: np.random.uniform(50, 200) for p in products}
for q in quarters:
    for p in products:
        sales[p] += np.random.uniform(-20, 40)
        rows.append({'quarter': q, 'product': p, 'sales': max(sales[p], 10)})
df = pd.DataFrame(rows)

fig = px.bar(df, x='sales', y='product',
             animation_frame='quarter',
             orientation='h',
             color='product',
             range_x=[0, 400],
             title='Product Sales Race by Quarter',
             labels={'sales': 'Cumulative Sales ($K)'},
             color_discrete_sequence=px.colors.qualitative.Bold)
fig.update_layout(showlegend=False, height=380)
fig.show()

### Real-World: Global CO2 Emissions Animation

> An environmental analyst animates per-capita CO2 emissions vs GDP across countries and decades for a policy presentation.

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

np.random.seed(10)
countries  = ['US','China','India','Germany','Brazil','UK','Japan','France','Canada','Australia']
continents = ['Americas','Asia','Asia','Europe','Americas','Europe','Asia','Europe','Americas','Oceania']
years      = list(range(1990, 2025, 5))

rows = []
for i, (c, cont) in enumerate(zip(countries, continents)):
    gdp  = np.random.uniform(5000, 60000)
    co2  = np.random.uniform(2, 16)
    pop  = np.random.uniform(20, 1400)
    for yr in years:
        gdp  *= np.random.uniform(1.01, 1.05)
        co2  *= np.random.uniform(0.97, 1.02)
        rows.append({'country':c,'continent':cont,'year':yr,
                     'gdp_pc':round(gdp,0),'co2_pc':round(co2,2),'pop':round(pop,1)})

df = pd.DataFrame(rows)

fig = px.scatter(
    df, x='gdp_pc', y='co2_pc',
    animation_frame='year', animation_group='country',
    size='pop', color='continent',
    hover_name='country',
    log_x=True, size_max=45,
    range_x=[3000, 120000], range_y=[0, 20],
    title='CO2 Emissions vs GDP per Capita (1990–2024)',
    labels={'gdp_pc': 'GDP per Capita ($, log)', 'co2_pc': 'CO2 per Capita (tonnes)'},
)
fig.update_layout(height=520)
fig.show()

## 9. 9. Maps & Geographic Charts

px.choropleth and px.scatter_geo create world or country-level maps. px.scatter_mapbox uses tile maps for city-level data.

**Choropleth world map**

In [ ]:
import plotly.express as px

df = px.data.gapminder().query("year == 2007")

fig = px.choropleth(
    df, locations='iso_alpha',
    color='lifeExp',
    hover_name='country',
    hover_data=['gdpPercap', 'pop'],
    color_continuous_scale='RdYlGn',
    range_color=[40, 85],
    title='Life Expectancy by Country (2007)',
    labels={'lifeExp': 'Life Expectancy'}
)
fig.update_layout(height=480, coloraxis_colorbar=dict(title='Years'))
fig.show()

**Scatter map — US cities**

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

np.random.seed(7)
cities = pd.DataFrame({
    'city':    ['New York','Los Angeles','Chicago','Houston','Phoenix',
                'Philadelphia','San Antonio','San Diego','Dallas','San Jose'],
    'lat':     [40.71,34.05,41.88,29.76,33.45,39.95,29.42,32.72,32.78,37.34],
    'lon':     [-74.01,-118.24,-87.63,-95.37,-112.07,-75.16,-98.49,-117.16,-96.80,-121.89],
    'revenue': np.random.uniform(50, 500, 10).round(1),
    'customers':np.random.randint(1000, 50000, 10),
})

fig = px.scatter_geo(cities, lat='lat', lon='lon',
                     size='revenue', color='revenue',
                     hover_name='city',
                     hover_data=['customers'],
                     color_continuous_scale='Reds',
                     scope='usa',
                     title='US Market Revenue by City')
fig.update_layout(height=430)
fig.show()

### Real-World: Global Sales Heatmap by Country

> A VP of Sales uses a choropleth to present YTD revenue performance by country and flag underperforming markets.

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

# ISO codes and country names
data = {
    'iso_alpha': ['USA','GBR','DEU','FRA','JPN','BRA','IND','AUS','CAN','MEX',
                  'CHN','KOR','SGP','ZAF','NLD','SWE','NOR','CHE','ITA','ESP'],
    'country':   ['US','UK','Germany','France','Japan','Brazil','India','Australia',
                  'Canada','Mexico','China','S.Korea','Singapore','S.Africa',
                  'Netherlands','Sweden','Norway','Switzerland','Italy','Spain'],
    'target':    [1000,300,280,220,350,150,200,180,260,120,
                  500,180,90,80,130,110,100,120,160,140],
}
np.random.seed(42)
df = pd.DataFrame(data)
df['actual']  = (df['target'] * np.random.uniform(0.6, 1.3, len(df))).round(0)
df['vs_target'] = ((df['actual'] - df['target']) / df['target'] * 100).round(1)

fig = px.choropleth(
    df, locations='iso_alpha',
    color='vs_target',
    hover_name='country',
    hover_data={'actual': True, 'target': True, 'vs_target': True},
    color_continuous_scale='RdYlGn',
    range_color=[-40, 40],
    title='YTD Revenue vs Target by Country (%)',
    labels={'vs_target': 'vs Target (%)'}
)
fig.update_layout(
    height=480,
    coloraxis_colorbar=dict(title='vs Target %', ticksuffix='%')
)
fig.show()

## 10. 10. Exporting & Sharing

Save charts as interactive HTML, static PNG/PDF/SVG (requires kaleido), or embed in web apps via fig.to_json().

**Export to HTML and JSON**

In [ ]:
import plotly.express as px
import os

df  = px.data.iris()
fig = px.scatter(df, x='sepal_length', y='sepal_width',
                 color='species', title='Iris — Export Demo')

# ── Standalone HTML (fully self-contained, shareable) ──
fig.write_html('iris_chart.html', include_plotlyjs='cdn')
print(f"HTML: {os.path.getsize('iris_chart.html') / 1024:.1f} KB")

# ── JSON (for embedding in web apps) ──
json_str = fig.to_json()
print(f"JSON length: {len(json_str):,} chars")

# ── Show in browser / Jupyter ──
fig.show()

# Cleanup
os.remove('iris_chart.html')

**Export as PNG/PDF with kaleido**

In [ ]:
import plotly.express as px
import os

# Note: requires  pip install kaleido
df  = px.data.tips()
fig = px.box(df, x='day', y='total_bill', color='time',
             title='Total Bill by Day')

try:
    fig.write_image('chart.png', width=800, height=500, scale=2)
    fig.write_image('chart.pdf')
    fig.write_image('chart.svg')
    print("Saved PNG, PDF, SVG")
    for f in ['chart.png','chart.pdf','chart.svg']:
        if os.path.exists(f): os.remove(f)
except Exception as e:
    print(f"kaleido not installed: {e}")
    print("Install with:  pip install kaleido")

fig.show()

### Real-World: Automated Interactive Report Generator

> A data engineer generates a self-contained HTML report with multiple charts embedded as a single shareable file.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import pandas as pd
import numpy as np
import os

np.random.seed(42)
months  = pd.date_range('2024-01', periods=12, freq='MS')
revenue = 100 + np.cumsum(np.random.randn(12) * 8) + np.arange(12) * 4
cac     = np.random.uniform(30, 80, 12)
churn   = np.random.uniform(0.02, 0.08, 12)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Monthly Revenue ($K)', 'Customer Acquisition Cost ($)',
                    'Churn Rate (%)', 'Revenue vs CAC'],
    specs=[[{},{}],[{},{}]]
)

fig.add_trace(go.Scatter(x=months, y=revenue.round(1),
                         fill='tozeroy', line=dict(color='#636EFA')), row=1, col=1)
fig.add_trace(go.Bar(x=months, y=cac.round(1),
                     marker_color='#EF553B'), row=1, col=2)
fig.add_trace(go.Scatter(x=months, y=(churn*100).round(2),
                         mode='lines+markers', line=dict(color='#AB63FA')), row=2, col=1)
fig.add_trace(go.Scatter(x=cac, y=revenue,
                         mode='markers', text=[m.strftime('%b') for m in months],
                         textposition='top center',
                         marker=dict(color='#00CC96', size=9)), row=2, col=2)

fig.update_layout(title='SaaS KPI Report — 2024', height=600,
                  showlegend=False, template='plotly_dark')

# Export as standalone HTML
out = 'saas_report.html'
fig.write_html(out, include_plotlyjs='cdn', full_html=True)
size_kb = os.path.getsize(out) / 1024
print(f"Report saved: {out} ({size_kb:.0f} KB)")
print("Open in any browser — fully interactive, no server needed.")
fig.show()
os.remove(out)